In [1]:
import kagglehub
import os

tikharm_32_path = kagglehub.dataset_download(
    "aryansraut/tikharm-data-32-frames"
)

path = kagglehub.model_download("arsalan9702/tikharm-models-new/pyTorch/default")

print("Path to model files:", path)

print("Dataset path:")
print(tikharm_32_path)

print("\nContents:")
for item in os.listdir(tikharm_32_path):
    print(item)

Path to model files: /kaggle/input/models/arsalan9702/tikharm-models-new/pytorch/default/1
Dataset path:
/kaggle/input/datasets/aryansraut/tikharm-data-32-frames

Contents:
TikHarm_frames_32
TikHarm_std
tikharm_metadata.csv
TikHarm_audio


In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchvision.transforms as T

from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

BASE = "/kaggle/input/datasets/aryansraut/tikharm-data-32-frames"

VISUAL_ROOT = f"{BASE}/TikHarm_frames_32/TikHarm_frames_32"
AUDIO_ROOT = f"{BASE}/TikHarm_audio/TikHarm_audio"

VISUAL_CKPT = "/kaggle/input/models/arsalan9702/tikharm-models-new/pytorch/default/1/best_swin3d_tikharm.pt"
AUDIO_CKPT  = "/kaggle/input/models/arsalan9702/tikharm-models-new/pytorch/default/1/best_audio_cnn14.pth"

NUM_FRAMES = 32
NUM_CLASSES = 4
MAX_AUDIO = 16000 * 10
BATCH_SIZE = 8

CLASSES = [
    "Adult Content",
    "Harmful Content",
    "Safe",
    "Suicide"
]

CLASS_TO_IDX = {
    cls: i for i, cls in enumerate(CLASSES)
}

print("Visual root:", VISUAL_ROOT)
print("Audio root:", AUDIO_ROOT)



Device: cuda
GPU: Tesla T4
Visual root: /kaggle/input/datasets/aryansraut/tikharm-data-32-frames/TikHarm_frames_32/TikHarm_frames_32
Audio root: /kaggle/input/datasets/aryansraut/tikharm-data-32-frames/TikHarm_audio/TikHarm_audio


In [3]:
!pip install torchlibrosa -q

In [4]:
from torchlibrosa.stft import Spectrogram, LogmelFilterBank


class CNN14(nn.Module):
    def __init__(self, classes_num=4):
        super().__init__()

        self.spectrogram_extractor = Spectrogram(
            n_fft=1024,
            hop_length=320,
            win_length=1024,
            window="hann",
            center=True,
            pad_mode="reflect"
        )

        self.logmel_extractor = LogmelFilterBank(
            sr=16000,
            n_fft=1024,
            n_mels=64,
            fmin=50,
            fmax=8000
        )

        self.bn0 = nn.BatchNorm2d(64)

        self.conv_block1 = self._conv_block(1, 64)
        self.conv_block2 = self._conv_block(64, 128)
        self.conv_block3 = self._conv_block(128, 256)
        self.conv_block4 = self._conv_block(256, 512)

        self.fc1 = nn.Linear(512, 512)
        self.fc_out = nn.Linear(512, classes_num)

    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)
        )

    def forward(self, x):
        x = self.spectrogram_extractor(x)
        x = self.logmel_extractor(x)

        x = x.transpose(1, 3)
        x = self.bn0(x)
        x = x.transpose(1, 3)

        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)

        x = torch.mean(x, dim=3)
        x = torch.mean(x, dim=2)

        x = F.relu(self.fc1(x))

        return self.fc_out(x)

In [5]:
class FusionDataset(Dataset):
    def __init__(
        self,
        visual_root,
        audio_root,
        split,
        num_frames=32,
        max_audio=16000 * 10
    ):
        self.samples = []
        self.num_frames = num_frames
        self.max_audio = max_audio

        self.transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        for cls in CLASSES:
            visual_class_dir = os.path.join(
                visual_root, split, cls
            )

            audio_class_dir = os.path.join(
                audio_root, split, cls
            )

            if not os.path.isdir(visual_class_dir):
                raise FileNotFoundError(
                    f"Missing directory: {visual_class_dir}"
                )

            for video_id in sorted(os.listdir(visual_class_dir)):
                video_dir = os.path.join(
                    visual_class_dir, video_id
                )

                if not os.path.isdir(video_dir):
                    continue

                audio_path = os.path.join(
                    audio_class_dir,
                    video_id + ".wav"
                )

                if os.path.isfile(audio_path):
                    self.samples.append(
                        (
                            video_dir,
                            audio_path,
                            CLASS_TO_IDX[cls]
                        )
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        video_dir, audio_path, label = self.samples[index]

        frame_files = sorted(
            f for f in os.listdir(video_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )

        if len(frame_files) == 0:
            raise RuntimeError(f"No frames found: {video_dir}")

        if len(frame_files) >= self.num_frames:
            indices = torch.linspace(
                0,
                len(frame_files) - 1,
                self.num_frames
            ).long()

            selected_frames = [
                frame_files[i] for i in indices
            ]
        else:
            selected_frames = (
                frame_files +
                [frame_files[-1]] *
                (self.num_frames - len(frame_files))
            )

        frames = []

        for frame_file in selected_frames:
            image = Image.open(
                os.path.join(video_dir, frame_file)
            ).convert("RGB")

            frames.append(self.transform(image))

        video = torch.stack(frames)
        video = video.permute(1, 0, 2, 3)

        waveform, sample_rate = torchaudio.load(audio_path)

        if sample_rate != 16000:
            waveform = torchaudio.functional.resample(
                waveform,
                sample_rate,
                16000
            )

        waveform = waveform.mean(dim=0)

        if waveform.shape[0] < self.max_audio:
            waveform = F.pad(
                waveform,
                (0, self.max_audio - waveform.shape[0])
            )
        else:
            waveform = waveform[:self.max_audio]

        return video, waveform, label

In [6]:
train_dataset = FusionDataset(
    VISUAL_ROOT,
    AUDIO_ROOT,
    "train",
    NUM_FRAMES,
    MAX_AUDIO
)

val_dataset = FusionDataset(
    VISUAL_ROOT,
    AUDIO_ROOT,
    "val",
    NUM_FRAMES,
    MAX_AUDIO
)

test_dataset = FusionDataset(
    VISUAL_ROOT,
    AUDIO_ROOT,
    "test",
    NUM_FRAMES,
    MAX_AUDIO
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

Train: 2761
Val: 396
Test: 790


In [7]:
video, audio, label = train_dataset[0]

print("Video shape:", video.shape)
print("Audio shape:", audio.shape)
print("Label:", label)

assert video.shape == (3, 32, 224, 224)
assert audio.shape[0] == MAX_AUDIO

print("32-frame dataset verified.")

Video shape: torch.Size([3, 32, 224, 224])
Audio shape: torch.Size([160000])
Label: 0
32-frame dataset verified.


In [8]:
visual_model = torch.hub.load(
    "pytorch/vision:v0.15.2",
    "swin3d_t",
    pretrained=False
)
visual_model.head = nn.Linear(visual_model.head.in_features, NUM_CLASSES)

visual_checkpoint = torch.load(VISUAL_CKPT, map_location=device)
visual_model.load_state_dict(visual_checkpoint["model_state_dict"])
visual_model = visual_model.to(device)

audio_model = CNN14(NUM_CLASSES)
audio_checkpoint = torch.load(AUDIO_CKPT, map_location=device)
audio_model.load_state_dict(audio_checkpoint)
audio_model = audio_model.to(device)

# Freeze everything first
for p in visual_model.parameters():
    p.requires_grad = False
for p in audio_model.parameters():
    p.requires_grad = False

# Unfreeze last stage of Swin3D-T.
# depths=[2,2,6,2] -> features = [stage0, merge0, stage1, merge1, stage2, merge2, stage3]
#                        index =    0       1       2       3       4       5       6
for p in visual_model.features[6].parameters():
    p.requires_grad = True
for p in visual_model.norm.parameters():
    p.requires_grad = True

# Unfreeze last conv block + fc1 of CNN14
for p in audio_model.conv_block4.parameters():
    p.requires_grad = True
for p in audio_model.fc1.parameters():
    p.requires_grad = True

visual_trainable = sum(p.numel() for p in visual_model.parameters() if p.requires_grad)
audio_trainable = sum(p.numel() for p in audio_model.parameters() if p.requires_grad)
print(f"Visual trainable params: {visual_trainable:,}")
print(f"Audio trainable params:  {audio_trainable:,}")


def set_train_mode():
    """Unfrozen submodules -> train(), frozen submodules stay eval()
    so their BatchNorm running stats and dropout are untouched."""
    visual_model.eval()
    visual_model.features[6].train()
    visual_model.norm.train()

    audio_model.eval()
    audio_model.conv_block4.train()
    audio_model.fc1.train()

    fusion_model.train()


def set_eval_mode():
    visual_model.eval()
    audio_model.eval()
    fusion_model.eval()

print("Backbones loaded. Last block of each unfrozen for fine-tuning.")

Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.15.2
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Visual trainable params: 14,298,960
Audio trainable params:  1,443,840
Backbones loaded. Last block of each unfrozen for fine-tuning.


In [9]:
def get_visual_features(x):
    x = visual_model.patch_embed(x)
    x = visual_model.pos_drop(x)

    for layer in visual_model.features:
        x = layer(x)

    x = visual_model.norm(x)

    # Swin3D output: B, T, H, W, C
    x = x.mean(dim=(1, 2, 3))

    return x


def get_audio_features(x):
    with torch.cuda.amp.autocast(enabled=False):
        x = x.float()
        x = audio_model.spectrogram_extractor(x)
        x = audio_model.logmel_extractor(x)

    x = x.transpose(1, 3)
    x = audio_model.bn0(x)
    x = x.transpose(1, 3)

    x = audio_model.conv_block1(x)
    x = audio_model.conv_block2(x)
    x = audio_model.conv_block3(x)
    x = audio_model.conv_block4(x)

    x = x.mean(dim=3)
    x = x.mean(dim=2)

    x = F.relu(audio_model.fc1(x))

    return x

In [10]:
with torch.no_grad():
    sample_video = video.unsqueeze(0).to(device)
    sample_audio = audio.unsqueeze(0).to(device)

    vf = get_visual_features(sample_video)
    af = get_audio_features(sample_audio)

print("Visual features:", vf.shape)
print("Audio features:", af.shape)

Visual features: torch.Size([1, 768])
Audio features: torch.Size([1, 512])


/tmp/ipykernel_228/958229649.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


In [11]:
class FeatureFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(768 + 512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 4)
        )

    def forward(self, visual_features, audio_features):
        x = torch.cat(
            [visual_features, audio_features],
            dim=1
        )

        return self.network(x)




fusion_model = FeatureFusion().to(device)
criterion = nn.CrossEntropyLoss()

BACKBONE_LR = 1e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4

backbone_params = (
    list(visual_model.features[6].parameters())
    + list(visual_model.norm.parameters())
    + list(audio_model.conv_block4.parameters())
    + list(audio_model.fc1.parameters())
)

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": fusion_model.parameters(), "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY
)

NUM_EPOCHS = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# GPU throughput settings
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GRAD_ACCUM_STEPS = 1  # raise to 2-4 if you hit CUDA OOM at BATCH_SIZE=8

/tmp/ipykernel_228/382721630.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


In [12]:
def train_one_epoch():
    set_train_mode()

    running_loss = 0.0
    correct = 0
    total = 0

    optimizer.zero_grad()
    progress = tqdm(train_loader, desc="Training")

    for step, (videos, audios, labels) in enumerate(progress):
        videos = videos.to(device, non_blocking=True)
        audios = audios.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            vf = get_visual_features(videos)
            af = get_audio_features(audios)
            logits = fusion_model(vf, af)
            loss = criterion(logits, labels) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(backbone_params + list(fusion_model.parameters()), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * GRAD_ACCUM_STEPS * labels.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        progress.set_postfix(loss=f"{loss.item() * GRAD_ACCUM_STEPS:.4f}")

    return running_loss / total, correct / total

In [13]:
def evaluate(loader):
    set_eval_mode()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for videos, audios, labels in loader:
            videos = videos.to(device, non_blocking=True)
            audios = audios.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                vf = get_visual_features(videos)
                af = get_audio_features(audios)
                logits = fusion_model(vf, af)
                loss = criterion(logits, labels)

            running_loss += loss.item() * labels.size(0)
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

In [14]:
NUM_EPOCHS = 10
PATIENCE = 3

best_val_acc = 0.0
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc = evaluate(val_loader)

    print()
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc:  {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val Acc:    {val_acc:.4f}")
    print(f"Backbone LR: {optimizer.param_groups[0]['lr']:.2e}")
    print("-" * 40)

    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        torch.save({
            "visual_state_dict": visual_model.state_dict(),
            "audio_state_dict": audio_model.state_dict(),
            "fusion_state_dict": fusion_model.state_dict(),
        }, "best_finetuned_32frames.pt")

        print("Best model saved.")
    else:
        patience_counter += 1
        print(f"No improvement: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print("Early stopping.")
        break

Training:   0%|          | 0/346 [00:00<?, ?it/s]/tmp/ipykernel_228/2559577969.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_228/958229649.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
Training: 100%|██████████| 346/346 [10:29<00:00,  1.82s/it, loss=0.0000]
/tmp/ipykernel_228/3324726577.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):



Epoch 1/10
Train Loss: 0.1328
Train Acc:  0.9714
Val Loss:   0.7714
Val Acc:    0.8914
Backbone LR: 1.00e-05
----------------------------------------
Best model saved.


Training: 100%|██████████| 346/346 [10:25<00:00,  1.81s/it, loss=0.0000]



Epoch 2/10
Train Loss: 0.0669
Train Acc:  0.9859
Val Loss:   0.8050
Val Acc:    0.8889
Backbone LR: 9.76e-06
----------------------------------------
No improvement: 1/3


Training: 100%|██████████| 346/346 [12:25<00:00,  2.16s/it, loss=0.0000]



Epoch 3/10
Train Loss: 0.0411
Train Acc:  0.9913
Val Loss:   0.9815
Val Acc:    0.8864
Backbone LR: 9.05e-06
----------------------------------------
No improvement: 2/3


Training: 100%|██████████| 346/346 [10:14<00:00,  1.78s/it, loss=0.0000]



Epoch 4/10
Train Loss: 0.0391
Train Acc:  0.9928
Val Loss:   1.0931
Val Acc:    0.8914
Backbone LR: 7.94e-06
----------------------------------------
No improvement: 3/3
Early stopping.


In [19]:
checkpoint = torch.load("best_finetuned_32frames.pt", map_location=device)
visual_model.load_state_dict(checkpoint["visual_state_dict"])
audio_model.load_state_dict(checkpoint["audio_state_dict"])
fusion_model.load_state_dict(checkpoint["fusion_state_dict"])

test_loss, test_acc = evaluate(test_loader)

print(f"Best Val Accuracy: {best_val_acc:.4f}")
print(f"Test Accuracy:     {test_acc:.4f}")

/tmp/ipykernel_228/3324726577.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_228/958229649.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


Best Val Accuracy: 0.8914
Test Accuracy:     0.8747


In [20]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score
)
import numpy as np


def classification_report_full(loader):
    set_eval_mode()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for videos, audios, labels in loader:
            videos = videos.to(device)
            audios = audios.to(device)

            vf = get_visual_features(videos)
            af = get_audio_features(audios)

            logits = fusion_model(vf, af)
            predictions = logits.argmax(dim=1)

            y_true.extend(labels.numpy())
            y_pred.extend(predictions.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    print("=" * 60)
    print("FEATURE FUSION — 32 FRAMES")
    print("=" * 60)

    print(
        f"Accuracy: {accuracy_score(y_true, y_pred):.4f}"
    )

    print(
        f"Balanced Accuracy: "
        f"{balanced_accuracy_score(y_true, y_pred):.4f}"
    )

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=CLASSES,
            digits=4
        )
    )

    print("Confusion Matrix:")
    print(
        confusion_matrix(y_true, y_pred)
    )


classification_report_full(test_loader)

/tmp/ipykernel_228/958229649.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


FEATURE FUSION — 32 FRAMES
Accuracy: 0.8747
Balanced Accuracy: 0.8748

Classification Report:
                 precision    recall  f1-score   support

  Adult Content     0.8670    0.9026    0.8844       195
Harmful Content     0.8922    0.7525    0.8164       198
           Safe     0.8950    0.8950    0.8950       200
        Suicide     0.8500    0.9492    0.8969       197

       accuracy                         0.8747       790
      macro avg     0.8761    0.8748    0.8732       790
   weighted avg     0.8762    0.8747    0.8732       790

Confusion Matrix:
[[176   9   3   7]
 [ 16 149  15  18]
 [  6   7 179   8]
 [  5   2   3 187]]
